## Exception Handling Done Right: Catching, Raising, and Communicating Errors Clearly

## Why Does Exception Handling Matter?

Poor exception handling is one of the most common sources of hard-to-debug code in student projects:

```python
# ❌ Swallows all errors — bugs disappear silently
try:
    result = process(data)
except:
    pass

# ❌ Too broad — catches things you didn't intend
try:
    result = process(data)
except Exception:
    print("Something went wrong")

# ✅ Specific, informative, recoverable
try:
    result = process(data)
except ValueError as e:
    print(f"Invalid input: {e}")
except FileNotFoundError as e:
    print(f"Missing file: {e}")
```

### Learning Goals
- Catch **specific** exceptions instead of bare `except` or broad `Exception`
- Write **meaningful error messages** that help the caller fix the problem
- Use `else` and `finally` blocks correctly
- Create **custom exception classes** for domain-specific errors
- Understand when to **catch**, when to **raise**, and when to **re-raise**

## Quick Reference

### Full try/except/else/finally structure
```python
try:
    result = risky_operation()
except SpecificError as e:   # runs if SpecificError is raised
    handle_error(e)
except AnotherError as e:    # runs if AnotherError is raised
    handle_other(e)
else:                        # runs ONLY if no exception was raised
    use_result(result)
finally:                     # ALWAYS runs (cleanup)
    cleanup()
```

### Common built-in exceptions
| Exception | When to use |
|---|---|
| `ValueError` | Argument has the right type but wrong value (`age = -5`) |
| `TypeError` | Argument is the wrong type entirely (`name = 42`) |
| `KeyError` | Missing key in a dict |
| `IndexError` | Index out of range in a list |
| `FileNotFoundError` | File or directory does not exist |
| `ZeroDivisionError` | Division by zero |
| `AttributeError` | Object does not have the expected attribute |
| `RuntimeError` | General error when no better type fits |

### Custom exception classes
```python
class InsufficientFundsError(ValueError):
    """Raised when an account balance is too low for a transaction."""

    def __init__(self, balance: float, amount: float):
        self.balance = balance
        self.amount  = amount
        super().__init__(
            f"Cannot withdraw {amount:.2f}: balance is only {balance:.2f}"
        )
```

### Golden rules
1. **Never use bare `except:`** — always name the exception
2. **Never silently `pass`** an exception unless you explicitly document why
3. **Catch only what you can handle** — let unexpected errors propagate
4. **Raise early, with context** — include the bad value in the message
5. **Use `else`** for code that only runs on success — not `try` itself

## Warm-up — Diagnose the Anti-patterns

Each snippet has at least one exception-handling problem.  
**In the markdown cell beneath each one, name the problem and describe the fix.**

In [ ]:
# Snippet A — the silent swallower
def load_settings(filepath):
    try:
        with open(filepath) as f:
            return f.read()
    except:
        pass

**Problem in Snippet A** (double-click to edit):

- What is wrong: 
- How to fix it: 

In [ ]:
# Snippet B — too much in the try block
def parse_and_display(raw):
    try:
        value = int(raw)
        doubled = value * 2
        label = f"Double of {raw} is {doubled}"
        print(label)
    except ValueError:
        print("Conversion failed")

**Problem in Snippet B**:

- What is wrong: 
- How to fix it: 

In [ ]:
# Snippet C — unhelpful error message
def set_age(age):
    if not isinstance(age, int) or age < 0:
        raise ValueError("Bad value")
    return age

**Problem in Snippet C**:

- What is wrong: 
- How to fix it: 

In [ ]:
# Snippet D — catching what you can't handle
import json

def load_json(filepath):
    try:
        with open(filepath) as f:
            return json.load(f)
    except Exception as e:
        print(f"Error: {e}")
        return {}

**Problem in Snippet D**:

- What is wrong: 
- How to fix it: 

## Exercise 1 — Catch the Right Exception

Replace bare or over-broad `except` blocks with specific exception types.  
Add a clear error message to each handler.

In [1]:
# Original 1a — bare except on a file operation
def read_file(path):
    try:
        with open(path) as f:
            return f.read()
    except:
        return None

print(read_file("nonexistent.txt"))

None


In [ ]:
# Your version of 1a — catch FileNotFoundError and PermissionError separately
def read_file(path: str) -> str | None:
    try:
        with open(path) as f:
            return f.read()
    # YOUR EXCEPT CLAUSES HERE

print(read_file("nonexistent.txt"))  # prints a clear message, returns None

In [ ]:
# Original 1b — multiple operations, one catch-all
import json

def parse_user_age(json_string):
    try:
        data = json.loads(json_string)
        age = int(data["age"])
        return age
    except Exception:
        return -1

print(parse_user_age('{"age": "25"}'))
print(parse_user_age('{"name": "Alice"}'))
print(parse_user_age('NOT JSON'))

In [ ]:
# Your version of 1b — three separate except clauses: json.JSONDecodeError, KeyError, ValueError
# Each should print a different, informative message before returning None
import json

def parse_user_age(json_string: str) -> int | None:
    try:
        data = json.loads(json_string)
        age = int(data["age"])
        return age
    # YOUR EXCEPT CLAUSES HERE

print(parse_user_age('{"age": "25"}'))    # 25
print(parse_user_age('{"name": "Alice"}')) # prints KeyError message, returns None
print(parse_user_age('NOT JSON'))          # prints JSON message, returns None

## Exercise 2 — else and finally

The `else` block runs **only if no exception was raised** in `try` — it's cleaner than putting success logic inside `try`.  
The `finally` block **always runs** — use it for cleanup.

Restructure each snippet to use `else` and/or `finally` correctly.

In [ ]:
# Original 2a — success logic mixed into the try block
def convert_and_square(value):
    try:
        number = float(value)
        result = number ** 2       # should only run if float() succeeded
        print(f"{value} squared is {result}")  # same here
        return result
    except ValueError:
        print(f"Cannot convert '{value}' to a number")
        return None

convert_and_square("4")
convert_and_square("abc")

In [ ]:
# Your version of 2a — move success logic to the `else` block
def convert_and_square(value: str) -> float | None:
    try:
        number = float(value)
    except ValueError:
        print(f"Cannot convert '{value}' to a number")
        return None
    else:
        # YOUR CODE HERE — this runs only when float() succeeded
        pass

convert_and_square("4")    # 4.0 squared is 16.0
convert_and_square("abc")  # Cannot convert 'abc' to a number

In [ ]:
# Original 2b — cleanup not guaranteed if an exception occurs mid-way
def process_file(path):
    f = open(path, "w")
    try:
        f.write("line 1\n")
        int("oops")         # this raises ValueError — file never closed!
        f.write("line 2\n")
    except ValueError as e:
        print(f"Write failed: {e}")
    f.close()  # not reached if exception was raised before this

process_file("/tmp/test_output.txt")

In [ ]:
# Your version of 2b
# Option A: use `finally` to guarantee f.close()
# Option B (better): use a `with` statement — it handles closing automatically
# Implement BOTH and add a comment explaining the difference.

# Option A — finally
def process_file_finally(path: str) -> None:
    # YOUR CODE HERE
    pass

# Option B — with statement
def process_file_with(path: str) -> None:
    # YOUR CODE HERE
    pass

process_file_finally("/tmp/test_a.txt")
process_file_with("/tmp/test_b.txt")

## Exercise 3 — Raise with Meaning

Good error messages tell the caller **what went wrong**, **what value caused it**, and **what is expected**.  

Improve each `raise` statement so it includes all three pieces of information.

In [ ]:
# Original 3 — vague raises
def create_user(username, age, email):
    if not username:
        raise ValueError("Error")
    if not isinstance(age, int):
        raise TypeError("Wrong type")
    if age < 0 or age > 130:
        raise ValueError("Out of range")
    if "@" not in email:
        raise ValueError("Invalid")
    return {"username": username, "age": age, "email": email}

try:
    create_user("", 25, "alice@example.com")
except ValueError as e:
    print(e)

In [ ]:
# Your version of Exercise 3
# Each raise should include: what the field is, what value was received, what is expected.
# Example: f"age must be an integer, got {type(age).__name__}"

def create_user(username: str, age: int, email: str) -> dict:
    # YOUR CODE HERE
    pass

tests = [
    ("",      25, "alice@example.com"),
    ("Alice", "x", "alice@example.com"),
    ("Alice", -1,  "alice@example.com"),
    ("Alice", 25,  "aliceexample.com"),
    ("Alice", 25,  "alice@example.com"),  # valid
]
for args in tests:
    try:
        print(create_user(*args))
    except (ValueError, TypeError) as e:
        print(f"{type(e).__name__}: {e}")

## Exercise 4 — Custom Exception Classes

Built-in exceptions are generic. Custom exception classes make your domain errors **self-documenting** and **catchable independently**.

In [ ]:
# Original 4 — everything is a plain ValueError
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise ValueError(f"Insufficient funds: tried to withdraw {amount}, balance is {self.balance}")
        self.balance -= amount

acc = BankAccount("Alice", 100.0)
try:
    acc.withdraw(200)
except ValueError as e:
    print(e)

In [ ]:
# Your version of Exercise 4
# 1. Create InvalidAmountError(ValueError) for negative/zero amounts
# 2. Create InsufficientFundsError(ValueError) — store .balance and .amount as attributes
# 3. Update deposit() and withdraw() to raise the specific custom exception
# 4. Show that callers can now catch each error independently

# YOUR CUSTOM EXCEPTION CLASSES HERE


class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        # YOUR CODE HERE
        pass

    def withdraw(self, amount: float) -> None:
        # YOUR CODE HERE
        pass

acc = BankAccount("Alice", 100.0)

# Test 1: catch InsufficientFundsError specifically
try:
    acc.withdraw(200)
except InsufficientFundsError as e:
    print(f"Caught InsufficientFundsError: {e}")
    print(f"  Attempted: {e.amount}, Available: {e.balance}")

# Test 2: catch InvalidAmountError specifically
try:
    acc.deposit(-50)
except InvalidAmountError as e:
    print(f"Caught InvalidAmountError: {e}")

## Hints

**Exercise 1a** — Two distinct problems can occur: the file doesn't exist (`FileNotFoundError`) or you don't have permission to read it (`PermissionError`). Handle them separately with different messages.

**Exercise 1b** — `json.loads` raises `json.JSONDecodeError`, dict access raises `KeyError`, and `int()` raises `ValueError`. Each is a different failure — handle each with a message that names which step failed.

**Exercise 2a** — Move `result = number ** 2` and the `print` into the `else` block. The `else` block only runs if `float(value)` succeeded — no extra `if` needed.

**Exercise 2b** — In Option A, open the file before `try`, then call `f.close()` in `finally`. In Option B, the `with` statement handles closing automatically, even if an exception is raised.

**Exercise 3** — A good message template: `"<field> must be <constraint>, got <repr(value)>"`. For example: `f"age must be between 0 and 130, got {age}"`.

**Exercise 4** — Inherit from `ValueError` so existing code catching `ValueError` still works. Store extra attributes (`self.amount`, `self.balance`) before calling `super().__init__(message)` so callers can read them programmatically.

## Model Solutions

In [ ]:
print("=== Warm-up Answers ===")
# A: bare `except` catches everything including KeyboardInterrupt and SystemExit.
#    Silently passing hides bugs completely. Fix: catch FileNotFoundError explicitly.

# B: The computation and print are inside `try` unnecessarily.
#    Only the line that can raise should be in `try`; success logic belongs in `else`.

# C: "Bad value" tells the caller nothing. Fix: include the actual value and expectation
#    e.g. f"age must be a non-negative int, got {age!r}"

# D: `except Exception` catches JSONDecodeError, KeyError, TypeError, etc. — too broad.
#    Fix: catch `FileNotFoundError` and `json.JSONDecodeError` separately.
#    Returning {} silently on a missing file hides the real problem from the caller.
print("See comments above")

In [ ]:
print("=== Exercise 1 ===")

# 1a
def read_file(path: str) -> str | None:
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        print(f"File not found: {path}")
        return None
    except PermissionError:
        print(f"Permission denied: {path}")
        return None

print(read_file("nonexistent.txt"))

# 1b
import json

def parse_user_age(json_string: str) -> int | None:
    try:
        data = json.loads(json_string)
        age = int(data["age"])
        return age
    except json.JSONDecodeError as e:
        print(f"Invalid JSON: {e}")
        return None
    except KeyError:
        print("Missing 'age' field in JSON")
        return None
    except ValueError as e:
        print(f"'age' could not be converted to int: {e}")
        return None

print(parse_user_age('{"age": "25"}'))
print(parse_user_age('{"name": "Alice"}'))
print(parse_user_age('NOT JSON'))

In [ ]:
print("=== Exercise 2 ===")

# 2a — else block
def convert_and_square(value: str) -> float | None:
    try:
        number = float(value)
    except ValueError:
        print(f"Cannot convert '{value}' to a number")
        return None
    else:
        result = number ** 2
        print(f"{value} squared is {result}")
        return result

convert_and_square("4")
convert_and_square("abc")

# 2b — finally vs with
def process_file_finally(path: str) -> None:
    f = open(path, "w")
    try:
        f.write("line 1\n")
        int("oops")           # raises ValueError
        f.write("line 2\n")
    except ValueError as e:
        print(f"Write failed: {e}")
    finally:
        f.close()             # always runs — file is always closed

# The `with` version is cleaner — context manager handles close() automatically
def process_file_with(path: str) -> None:
    with open(path, "w") as f:
        try:
            f.write("line 1\n")
            int("oops")
            f.write("line 2\n")
        except ValueError as e:
            print(f"Write failed: {e}")

process_file_finally("/tmp/test_a.txt")
process_file_with("/tmp/test_b.txt")

In [ ]:
print("=== Exercise 3 ===")

def create_user(username: str, age: int, email: str) -> dict:
    if not username:
        raise ValueError("username must be a non-empty string")
    if not isinstance(age, int):
        raise TypeError(f"age must be an integer, got {type(age).__name__!r}")
    if age < 0 or age > 130:
        raise ValueError(f"age must be between 0 and 130, got {age}")
    if "@" not in email:
        raise ValueError(f"email must contain '@', got {email!r}")
    return {"username": username, "age": age, "email": email}

tests = [
    ("",      25, "alice@example.com"),
    ("Alice", "x", "alice@example.com"),
    ("Alice", -1,  "alice@example.com"),
    ("Alice", 25,  "aliceexample.com"),
    ("Alice", 25,  "alice@example.com"),
]
for args in tests:
    try:
        print(create_user(*args))
    except (ValueError, TypeError) as e:
        print(f"{type(e).__name__}: {e}")

In [ ]:
print("=== Exercise 4 ===")

class InvalidAmountError(ValueError):
    """Raised when a transaction amount is zero or negative."""
    def __init__(self, amount: float):
        self.amount = amount
        super().__init__(f"Transaction amount must be positive, got {amount}")

class InsufficientFundsError(ValueError):
    """Raised when a withdrawal exceeds the available balance."""
    def __init__(self, balance: float, amount: float):
        self.balance = balance
        self.amount  = amount
        super().__init__(
            f"Cannot withdraw {amount:.2f}: balance is only {balance:.2f}"
        )

class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0):
        self.owner   = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise InvalidAmountError(amount)
        self.balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise InvalidAmountError(amount)
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount

acc = BankAccount("Alice", 100.0)

try:
    acc.withdraw(200)
except InsufficientFundsError as e:
    print(f"Caught InsufficientFundsError: {e}")
    print(f"  Attempted: {e.amount}, Available: {e.balance}")

try:
    acc.deposit(-50)
except InvalidAmountError as e:
    print(f"Caught InvalidAmountError: {e}")

## Bonus — Refactor a Real-world Handler

The function below simulates loading and validating a user config file.  
It has **five** exception-handling problems. Find them all and rewrite the function cleanly.

Problems to find:
1. Bare `except`
2. Silent `pass`
3. Overly broad catch in the wrong place
4. Success logic mixed into `try`
5. Vague error message

In [ ]:
# --- ORIGINAL ---
import json

def load_user_config(filepath):
    try:
        f = open(filepath)
        raw = f.read()
        f.close()
        config = json.loads(raw)
        username = config["username"]
        age      = int(config["age"])
        print(f"Loaded config for {username}, age {age}")
        return config
    except Exception:
        pass
    except:
        return None

In [ ]:
# --- YOUR CLEAN VERSION ---
import json

def load_user_config(filepath: str) -> dict | None:
    """Load and validate a user configuration file.

    Args:
        filepath: Path to a JSON file containing 'username' (str) and 'age' (int).

    Returns:
        The parsed config dict, or None if the file cannot be loaded.

    Raises:
        FileNotFoundError: If the file does not exist.
        json.JSONDecodeError: If the file is not valid JSON.
        KeyError: If required keys are missing.
        ValueError: If 'age' cannot be converted to an integer.
    """
    # YOUR CODE HERE
    pass